# 01 - Python Fundamentals: Syntax, Types & Control Flow

Part of the Python, DSA & Git chapter. This is the "make sure nothing basic trips you up under pressure" notebook - everything here should eventually be automatic, because DSA interviews punish hesitation on fundamentals just as much as not knowing the algorithm.

Covers: variables & dynamic typing, numeric types, strings, booleans/truthiness, type conversion, control flow, loops. Practice exercises at the end.

## Variables & dynamic typing

Python is dynamically typed (a variable's type is determined at runtime, and can change) but strongly typed (it won't silently coerce incompatible types, e.g. "2" + 2 raises TypeError - unlike JavaScript). A variable name is just a label pointing at an object in memory; assignment doesn't copy, it binds a name to an object.

In [1]:
x = 5
print(type(x), x)
x = "now a string"          # totally legal -- x is rebound to a new object
print(type(x), x)

# Multiple assignment & unpacking
a, b, c = 1, 2, 3
print(a, b, c)

a, b = b, a                 # classic Python swap -- no temp variable needed
print("after swap:", a, b)

# id() shows this is about object identity, not "boxes with values"
y = x
print("x and y point to the same object:", x is y, id(x) == id(y))

<class 'int'> 5
<class 'str'> now a string
1 2 3
after swap: 2 1
x and y point to the same object: True True


## Numeric types & operators

Python has int (arbitrary precision - no overflow, unlike fixed-width ints in C/Java), float (IEEE-754 double), and complex. Know the operators cold, especially the ones that trip people up: / is always float division, // is floor division, % is modulo, ** is exponentiation.

In [2]:
print(7 / 2)     # 3.5   -- true division, always returns float
print(7 // 2)    # 3     -- floor division
print(-7 // 2)   # -4    -- floors toward NEGATIVE infinity, not toward zero (a classic gotcha)
print(7 % 2)     # 1     -- modulo
print(-7 % 2)    # 1     -- sign follows the DIVISOR in Python (different from C/Java!)
print(2 ** 10)   # 1024  -- exponentiation
print(10_000_000)  # underscores for readability in large literals, ignored by the parser

# Arbitrary precision -- this does NOT overflow, unlike int64 in most other languages
print(2 ** 100)

# Operator precedence: ** binds tighter than unary minus on its left operand
print(-2 ** 2)   # -4, not 4 -- equivalent to -(2**2)
print((-2) ** 2)  # 4

3.5
3
-4
1
1
1024
10000000
1267650600228229401496703205376
-4
4


## Strings

Strings are immutable sequences of Unicode characters - every "modification" actually creates a new string object. In principle this makes repeated concatenation (s = s + x in a loop) O(n-squared) overall, since each + allocates a new, longer string. CPython specifically optimizes the common case (s += x, where s has no other live references) into an in-place buffer resize - which is why a naive benchmark below won't always show the blowup. That optimization is a CPython implementation detail, not a language guarantee: it silently stops applying the moment another reference to the string exists (e.g. you stored an intermediate copy somewhere), and other implementations (PyPy, MicroPython) don't provide it at all. ''.join(...) is O(n) unconditionally, which is why it's the recommended pattern regardless of what a quick benchmark seems to show.

In [3]:
s = "Machine Learning"

print(s[0], s[-1])          # indexing, negative indices count from the end
print(s[0:7])                # slicing: [start:stop) -- stop is exclusive
print(s[::-1])               # reverse a string -- extremely common interview trick
print(s[::2])                # every 2nd character

print(s.lower(), s.upper())
print(s.split(" "))          # splits on spaces -> list of words
print("-".join(["a", "b", "c"]))   # joins with a separator -> "a-b-c"
print(s.replace("Learning", "Learning (ML)"))
print("  padded  ".strip())  # remove leading/trailing whitespace

name, score = "Alex", 97.456
print(f"{name} scored {score:.1f}%")     # f-strings: the modern, preferred way to format
print("{} scored {:.1f}%".format(name, score))  # .format() -- still seen in older code

# Strings are immutable -- this creates a NEW string, doesn't mutate s
s2 = s.replace("Machine", "Deep")
print(s, "->", s2)

# Why repeated += concatenation is O(n^2) IN PRINCIPLE -- and why a naive
# benchmark of it can be misleading in CPython specifically.
import timeit

def concat_simple(n):
    out = ""
    for i in range(n):
        out += "x"
    return out                              # CPython's fast path often applies here

def concat_with_extra_ref(n):
    out = ""
    history = []
    for i in range(n):
        out += "x"
        history.append(out)                 # keeps old copies alive -> defeats the fast path
    return out

def good_concat(n):
    return "".join("x" for _ in range(n))   # O(n) unconditionally

n = 8000
t_simple = timeit.timeit(lambda: concat_simple(n), number=5)
t_with_ref = timeit.timeit(lambda: concat_with_extra_ref(n), number=5)
t_good = timeit.timeit(lambda: good_concat(n), number=5)
print("+=  (CPython fast path applies)     :", round(t_simple, 4), "s")
print("+=  (fast path defeated, true O(n^2)):", round(t_with_ref, 4), "s  <- notice the blowup")
print("''.join(...)  (O(n) always)          :", round(t_good, 4), "s")

M g
Machine
gninraeL enihcaM
McieLann
machine learning MACHINE LEARNING
['Machine', 'Learning']
a-b-c
Machine Learning (ML)
padded
Alex scored 97.5%
Alex scored 97.5%
Machine Learning -> Deep Learning
+=  (CPython fast path applies)     : 0.0027 s
+=  (fast path defeated, true O(n^2)): 0.1171 s  <- notice the blowup
''.join(...)  (O(n) always)          : 0.0022 s


## Booleans, truthiness & short-circuit evaluation

Every object in Python has an implicit boolean value. Knowing exactly what's "falsy" is a small thing that saves real debugging time: 0, 0.0, "", [], {}, set(), (), and None are all falsy - everything else is truthy, including things people sometimes assume are falsy (the string "0", or a non-empty list containing 0).

In [4]:
falsy_examples = [0, 0.0, "", [], {}, set(), (), None]
truthy_examples = ["0", " ", [0], {0: 0}, -1, 0.0001]

print("Falsy:", [bool(x) for x in falsy_examples])
print("Truthy:", [bool(x) for x in truthy_examples])

# `and` / `or` return one of their OPERANDS, not just True/False -- and they short-circuit
print(0 or "default")        # -> "default"  (0 is falsy, so evaluate/return the right side)
print("value" or "default")  # -> "value"    (left side is truthy, right side never evaluated)
print("a" and "b")           # -> "b"        (both truthy -> returns the LAST one)
print("" and "b")            # -> ""         (left is falsy -> short-circuits, returns left)

# This pattern is extremely common for default values
def greet(name=None):
    name = name or "stranger"
    return f"Hello, {name}!"

print(greet())
print(greet("Ada"))

# Ternary expression
x = 7
label = "even" if x % 2 == 0 else "odd"
print(label)

Falsy: [False, False, False, False, False, False, False, False]
Truthy: [True, True, True, True, True, True]
default
value
b

Hello, stranger!
Hello, Ada!
odd


## Type conversion

Explicit conversion functions: int(), float(), str(), bool(), list(), tuple(), set(). Watch for the common failure mode - int("3.5") raises ValueError (it only parses clean integer strings); go through float first if you're not sure.

In [5]:
print(int("42"), float("3.14"), str(42), bool(1))
print(int(float("3.9")))   # 3 -- int() truncates toward zero, does NOT round

try:
    int("3.5")
except ValueError as e:
    print("int('3.5') fails:", e, "-- go through float() first:", int(float("3.5")))

print(list("abc"))          # string -> list of characters
print(list((1, 2, 3)))      # tuple -> list
print(set([1, 1, 2, 2, 3])) # list -> set, dedups automatically

42 3.14 42 True
3
int('3.5') fails: invalid literal for int() with base 10: '3.5' -- go through float() first: 3
['a', 'b', 'c']
[1, 2, 3]
{1, 2, 3}


## Control flow: if / elif / else

Python has no switch statement in versions before 3.10 (3.10+ has match/case - structural pattern matching, worth knowing exists even if not everywhere yet). Indentation is syntactically significant - it IS the block delimiter, not just a style convention.

In [6]:
def classify(n):
    if n < 0:
        return "negative"
    elif n == 0:
        return "zero"
    elif n % 2 == 0:
        return "positive even"
    else:
        return "positive odd"

for n in [-3, 0, 4, 7]:
    print(n, "->", classify(n))

# match/case (Python 3.10+) -- structural pattern matching, good to recognize
def describe(x):
    match x:
        case 0:
            return "zero"
        case int() | float() if x < 0:
            return "negative number"
        case [a, b]:
            return f"a 2-element list: {a}, {b}"
        case {"type": t}:
            return f"a dict with type={t}"
        case _:
            return "something else"

for val in [0, -5, [1, 2], {"type": "user"}, "hi"]:
    print(val, "->", describe(val))

-3 -> negative
0 -> zero
4 -> positive even
7 -> positive odd
0 -> zero
-5 -> negative number
[1, 2] -> a 2-element list: 1, 2
{'type': 'user'} -> a dict with type=user
hi -> something else


## Loops: for, while, range, enumerate, zip

for in Python iterates over any iterable (not "from 0 to n" like C) - that distinction matters a lot once you get to iterators/generators (next notebook). range(start, stop, step) is the tool for classic index-based loops. enumerate and zip are the two most-used loop helpers you should default to instead of manual indexing.

In [7]:
# range: stop is exclusive, just like slicing
print(list(range(5)))          # [0, 1, 2, 3, 4]
print(list(range(2, 10, 2)))   # [2, 4, 6, 8]
print(list(range(5, 0, -1)))   # [5, 4, 3, 2, 1]

# enumerate -- avoid manual index counters
names = ["alice", "bob", "carol"]
for i, name in enumerate(names):
    print(i, name)
for i, name in enumerate(names, start=1):   # custom start index
    print(i, name)

# zip -- iterate multiple sequences in lockstep, stops at the SHORTEST one
scores = [91, 85, 77]
for name, score in zip(names, scores):
    print(f"{name}: {score}")

# while, with break/continue
n = 0
total = 0
while n < 10:
    n += 1
    if n % 2 == 0:
        continue     # skip even numbers
    if n > 7:
        break        # stop early
    total += n
print("total:", total)

# The for/while ...else clause -- a genuinely Python-specific feature, worth
# knowing exists: the else block runs only if the loop completed WITHOUT hitting break.
def has_pair_summing_to(nums, target):
    seen = set()
    for x in nums:
        if target - x in seen:
            print("found pair summing to", target)
            break
        seen.add(x)
    else:
        print("no pair sums to", target)

has_pair_summing_to([2, 7, 11, 15], 9)
has_pair_summing_to([2, 7, 11, 15], 100)

[0, 1, 2, 3, 4]
[2, 4, 6, 8]
[5, 4, 3, 2, 1]
0 alice
1 bob
2 carol
1 alice
2 bob
3 carol
alice: 91
bob: 85
carol: 77
total: 16
found pair summing to 9
no pair sums to 100


## Practice exercises

Fill in each TODO, then run the check cell below to self-check. These are deliberately small - the goal is fluency, not difficulty (harder problems live in the DSA notebooks).

In [8]:
def is_palindrome(s):
    # Return True if s reads the same forwards and backwards (case-sensitive).
    # TODO: implement in one line using slicing
    raise NotImplementedError

def fizzbuzz(n):
    # Return a list of strings 1..n: multiples of 3 -> "Fizz", of 5 -> "Buzz",
    # of both -> "FizzBuzz", else the number as a string.
    # TODO: implement
    raise NotImplementedError

def count_vowels(s):
    # Count vowels (a,e,i,o,u, case-insensitive) in s.
    # TODO: implement
    raise NotImplementedError

def swap_case_manual(s):
    # Return s with each letter's case swapped, WITHOUT using str.swapcase().
    # TODO: implement
    raise NotImplementedError

In [9]:
def _check(name, fn, cases):
    for args, expected in cases:
        try:
            result = fn(args)
        except NotImplementedError:
            print("  [SKIP]", name, "-- not implemented yet")
            return
        except Exception as e:
            print("  [ERROR]", name, args, "--", e)
            return
        ok = result == expected
        tag = "PASS" if ok else "FAIL"
        extra = "" if ok else ("expected " + repr(expected))
        print("  [" + tag + "]", name + "(" + repr(args) + ") ->", repr(result), extra)

_check("is_palindrome", is_palindrome, [("racecar", True), ("hello", False), ("a", True)])
_check("fizzbuzz(5)", fizzbuzz, [(5, ["1", "2", "Fizz", "4", "Buzz"])])
_check("count_vowels", count_vowels, [("hello world", 3), ("xyz", 0)])
_check("swap_case_manual", swap_case_manual, [("Hello World", "hELLO wORLD")])

  [SKIP]

 is_palindrome -- not implemented yet
  [SKIP] fizzbuzz(5) -- not implemented yet
  [SKIP] count_vowels -- not implemented yet
  [SKIP] swap_case_manual -- not implemented yet


## Self-check before moving on

- [ ] I can explain the difference between / and //, and how % behaves with negative numbers
- [ ] I know why string concatenation with += in a loop is O(n-squared) and can fix it with .join()
- [ ] I can list what's falsy in Python without hesitating
- [ ] I understand or/and return operands, not just booleans, and can use that for defaults
- [ ] I default to enumerate/zip instead of manual index tracking in loops

Next: 02-data-structures-native.ipynb